In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scite.beam.continuous.continuous_beam_var_cs import ContinuousBeamVarCSModes
from dataclasses import replace as _replace


## Parametric study: $\theta_{pl}$ vs $x/d$

**Goal**: collect inelastic rotation $\theta_{pl}$ from the variable-CS continuous beam model
as a function of the neutral-axis depth ratio $x/d$ at the hogging section.

**Setup**:
- Rectangular 300 × 700 mm section, C30/37, B500B, EC2 design strength values.
- **Symmetric reinforcement**: $A_{s,top} = A_{s,bot} = A_s$ varied over five values.
- Variable-CS model: hogging zone uses top bars only, sagging zone uses bottom bars only
  → x/d at hogging matches the EC2 singly-reinforced formula.
- Equal spans $L = 6\,\text{m}$.

The $A_s$ values are derived from the target $x/d$ range $[0.02,\,0.50]$ via the
EC2 rectangular stress-block formula:
$$A_s = \frac{x/d \cdot 0.8 \cdot f_{cd}}{f_{yd}} \cdot b \cdot d$$

In [ ]:
# ── Geometry ──────────────────────────────────────────────────────────────────
b, h    = 300.0, 700.0
z_s_bot = 50.0
z_s_top = h - z_s_bot      # 650 mm from bottom
d_hog   = z_s_top           # hogging effective depth: bottom → top tension bars
L       = 6000.0

# ── Design material values ────────────────────────────────────────────────────
f_ck, f_yk   = 30.0, 500.0
gamma_c, gamma_s, alpha_cc = 1.5, 1.15, 0.85
f_cd = alpha_cc * f_ck / gamma_c   # = 17.0 MPa
f_yd = f_yk / gamma_s              # ≈ 434.78 MPa

# ── A_s range derived from target x/d ─────────────────────────────────────────
# x/d = omega/0.8  with  omega = rho * f_yd / f_cd  →  A_s = x/d * 0.8 * (f_cd/f_yd) * b * d
xd_targets = np.linspace(0.05, 0.50, 7)
A_s_range  = xd_targets * 0.8 * (f_cd / f_yd) * b * d_hog

print(f'Design:  f_cd = {f_cd:.2f} MPa,  f_yd = {f_yd:.2f} MPa,  d_hog = {d_hog:.0f} mm')
print()
print(f'{"x/d target":>12}  {"A_s [mm²]":>12}')
for xd, As in zip(xd_targets, A_s_range):
    print(f'{xd:12.3f}  {As:12.0f}')

## Parametric sweep

In [ ]:
results = {'design': [], 'mean': []}

for A_s in A_s_range:
    triple = ContinuousBeamVarCSModes.for_rc(
        b=b, h=h,
        A_s_bot=A_s, z_s_bot=z_s_bot,
        A_s_top=A_s, z_s_top=z_s_top,
        f_ck=f_ck, f_yk=f_yk,
        L_a=L, L_b=L,
        n_kappa=100,
    )
    print(f'A_s = {A_s:.0f} mm²')
    for mode_name, cba in [('design', triple.design), ('mean', triple.mean)]:
        xd           = cba.xd_hog_u
        theta_pl     = cba.theta_pl_at_step(-1)     * 1e3
        theta_pl_ec2 = cba.theta_pl_ec2_at_step(-1) * 1e3
        delta_mid    = cba.midspan_deflection_at_step(-1)   # mm
        q_u          = cba.q_hog_u
        lambda_sl    = cba.lambda_sl_at_step(d=d_hog)       # geometric shear slenderness
        k_lambda     = np.sqrt(lambda_sl / 3.0)             # EC2 §5.6.3 correction factor
        results[mode_name].append(dict(A_s=A_s, xd=xd,
                                       theta_pl=theta_pl,
                                       theta_pl_ec2=theta_pl_ec2,
                                       delta_mid=delta_mid,
                                       q_u=q_u,
                                       lambda_sl=lambda_sl,
                                       k_lambda=k_lambda))
        print(f'  {mode_name:12s}  x/d={xd:.4f}  '
              f'θ_pl={theta_pl:.2f}/{theta_pl_ec2:.2f} mrad (full/±0.6h)  '
              f'δ_mid={delta_mid:.1f} mm  q={q_u:.1f} kN/m  '
              f'λ={lambda_sl:.2f}  k={k_lambda:.3f}')
    print()


## $\theta_{pl}$ vs $x/d$ — full hogging zone vs EC2 1.2h zone, design vs mean

In [ ]:
COLOR = {'design': '#d62728', 'mean': '#2ca02c'}

fig, (ax_th, ax_lam, ax_d) = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

for mode in ('design', 'mean'):
    r      = results[mode]
    xd     = np.array([pt['xd']           for pt in r])
    th     = np.array([pt['theta_pl']     for pt in r])
    th_ec  = np.array([pt['theta_pl_ec2'] for pt in r])
    delta  = np.array([pt['delta_mid']    for pt in r])
    lam    = np.array([pt['lambda_sl']    for pt in r])
    c      = COLOR[mode]

    ax_th.plot(xd, th,    'o-',  color=c, lw=2,   ms=7,
               label=fr'$\theta_{{pl}}$ — {mode}, full zone')
    ax_th.plot(xd, th_ec, 's--', color=c, lw=1.5, ms=6,
               label=fr'$\theta_{{pl,EC2}}$ — {mode}, $\pm 0.6h$')
    ax_lam.plot(xd, lam, 'D-',  color=c, lw=2, ms=6,
                label=f'$\lambda$ — {mode}')
    ax_d.plot(xd,  delta, 'o-',  color=c, lw=2,   ms=7,
              label=f'$\delta_{{mid}}$ — {mode}')

ax_lam.axhline(3.0, color='gray', lw=1.2, ls=':', label=r'$\lambda = 3$ (Bild 5.6DE reference)')
ax_lam.text(0.48, 3.05, r'$\lambda = 3$', ha='right', va='bottom', fontsize=9, color='gray')

ax_th.set_ylabel(r'$\theta_{pl}$ [mrad]', fontsize=11)
ax_th.set_title(
    r'Symmetric $A_s$: var CS, design vs mean — rotation, slenderness, deflection',
    fontsize=10)
ax_th.grid(True, alpha=0.3, linestyle='--')
ax_th.legend(fontsize=9, ncol=2)

ax_lam.set_ylabel(r'$\lambda = a_s / d$  (EC2 §5.6.3)', fontsize=11)
ax_lam.grid(True, alpha=0.3, linestyle='--')
ax_lam.legend(fontsize=9)

ax_d.set_xlabel(r'$x/d$ — hogging section (top bars only)', fontsize=11)
ax_d.set_ylabel(r'$\delta_{mid}$ [mm]  (ULS load)', fontsize=11)
ax_d.grid(True, alpha=0.3, linestyle='--')
ax_d.legend(fontsize=9)

plt.tight_layout()
plt.show()


## Interpretation

**Four curves, two contrasts**:

| Pair | What it reveals |
|------|----------------|
| **Full zone vs EC2 ±0.6h** (solid vs dashed, same colour) | How much of the plastic rotation comes from outside the 1.2h plastic-hinge assumption. The full-zone integral grows with x/d because a heavier beam carries a higher q_u, pushing the hogging zone further into the span. EC2's fixed 1.2h window removes this effect. |
| **Design vs mean** (red vs green, same linestyle) | How material safety factors shift the curve. Design values (γ_c=1.5, γ_s=1.15, α_cc=0.85) give lower f_cd and f_yd → smaller x/d for the same A_s, and less curvature capacity κ_R → smaller θ_pl. Mean values (f_cm, f_yk·1.1) recover capacity closer to the laboratory. |

**Trend with x/d**:
- Full-zone θ_pl increases with x/d because the higher load q_u extends the hogging zone; the integration length grows faster than κ_R shrinks.
- EC2-zone θ_pl decreases with x/d, in line with the EC2 §5.6.3 Table 5.6N trend, because the fixed ±0.6h window sees only the local curvature — and κ_R does decrease as x/d increases (more steel → more compression depth → less ductility).

**Design vs mean gap**: The horizontal shift in x/d between red and green for the same A_s reflects the ratio of design to mean material strengths. The vertical gap in θ_pl reflects reduced curvature ductility under design strengths.

## Shear slenderness $\lambda$ and the EC2 reference (§5.6.3)

EC2 §5.6.3 defines the **shear slenderness** $\lambda$ as the ratio of the
distance from the zero-moment point (inflection point) to the maximum-moment
section (interior support) to the effective depth $d$:

$$\lambda = \frac{a_s}{d}$$

where $a_s$ is measured on the **true continuous beam** after redistribution —
not on an equivalent simply supported span.

For practical application without computing the moment profile, EC2 permits the
simplified formula (Eq. 5.12N):

$$\lambda \approx \frac{M_{Ed}}{V_{Ed} \cdot d}$$

where $M_{Ed}$ and $V_{Ed}$ are the bending moment and shear force at the
plastic-hinge section (interior support).  For a beam under UDL, the geometric
definition gives values roughly 20–25 % higher than the simplified formula.

**Correction factor for Bild 5.6DE** — The reference chart gives the basic
allowable plastic rotation $\theta_{pl,d}$ at $\lambda = 3.0$.
For other slenderness values, EC2 §5.6.3(4) requires multiplication by
(Eq. 5.11N):

$$k = \sqrt{\frac{\lambda}{3}}$$

**Effect on our L = 6 m comparison** — At L = 6 m with d = 650 mm the
geometric shear span gives $\lambda \approx 2.0$–$2.5$, so $k < 1$ and
the corrected allowable rotation is *below* the Bild 5.6DE reference curve.
A direct comparison would therefore overstate the available capacity.

**At L = 8 m** (same cross-section), $\lambda \approx 3.0$ and Bild 5.6DE
applies without correction ($k \approx 1$), making the comparison meaningful.
The subplot above shows the computed $\lambda$ vs $x/d$ for L = 6 m with a
reference line at $\lambda = 3$.


## Study 2: fixed sagging reinforcement, varying hogging reinforcement

**Motivation**: In Study 1, $A_{s,bot} = A_{s,top}$ (symmetric).  
Varying $A_s$ simultaneously changes both the hogging and the sagging zone M-κ curves.  
This study decouples the two: $A_{s,bot}$ is held **fixed** at a mid-range value while  
only $A_{s,top}$ is varied.

This isolates the effect of the hogging section depth $x/d$ on $\theta_{pl}$ from any  
change in the sagging zone capacity (which affects $q_u$ and the extent of the hogging  
zone integration window).

**Expected**: if the sagging reinforcement drives the θ_pl–x/d trend, the curves will  
look different here; if not, Study 2 reproduces Study 1's shape.

### What EC2 actually checks

EC2 §5.6.3 gives a *plastic rotation capacity* $\theta_{pl,d}$ as a function of $x/d$  
(Table 5.6N, for ductility class B/C and concrete C20–C90).  The designer's workflow is:

1. **Design the structure** with redistributed moments (say −20 % at the support, $\delta = 0.8$).  
2. **Compute the required plastic rotation** $\theta_{req}$ at the hinge from kinematics  
   (for a two-span beam: proportional to $\Delta M / EI \cdot L$).  
3. **Read off** $\theta_{pl,d}(x/d)$ from Table 5.6N for the chosen concrete and ductility class.  
4. **Check** $\theta_{req} \leq \theta_{pl,d}$.

So EC2's table is an *allowable capacity*, derived empirically (and validated against  
test data) — it represents a *lower-bound* plastic rotation capacity for sections with  
a given $x/d$, integrated over a $1.2h$ hinge zone.  
Our simulation computes the same integral from first principles, so comparing the two  
directly tests the conservatism of the EC2 empirical fit relative to the M-κ model.

Slenderness ($L/d$) enters the *demand* side ($\theta_{req}$), not the capacity table;  
so it cannot explain a discrepancy between our computed capacity and EC2's table values.

In [ ]:
# Fixed sagging reinforcement: A_s_bot stays at the middle value of Study 1
A_s_bot_fixed = float(A_s_range[len(A_s_range) // 2])   # ~mid-range sagging area

# Hogging reinforcement varied over the same x/d-derived range
A_s_top_range = xd_targets * 0.8 * (f_cd / f_yd) * b * d_hog

print(f'Fixed sagging A_s_bot = {A_s_bot_fixed:.0f} mm²  '
      f'(x/d_sag ≈ {A_s_bot_fixed / (b * d_hog) * f_yd / f_cd / 0.8:.3f})')
print()
print(f'{"x/d target":>12}  {"A_s_top [mm²]":>14}')
for xd, As in zip(xd_targets, A_s_top_range):
    print(f'{xd:12.3f}  {As:14.0f}')

results2 = {'design': [], 'mean': []}

for A_s_top in A_s_top_range:
    triple = ContinuousBeamVarCSModes.for_rc(
        b=b, h=h,
        A_s_bot=A_s_bot_fixed, z_s_bot=z_s_bot,
        A_s_top=A_s_top,       z_s_top=z_s_top,
        f_ck=f_ck, f_yk=f_yk,
        L_a=L, L_b=L,
        n_kappa=100,
    )
    print(f'\nA_s_top = {A_s_top:.0f} mm²')
    for mode_name, cba in [('design', triple.design), ('mean', triple.mean)]:
        xd           = cba.xd_hog_u
        theta_pl     = cba.theta_pl_at_step(-1)     * 1e3
        theta_pl_ec2 = cba.theta_pl_ec2_at_step(-1) * 1e3
        delta_mid    = cba.midspan_deflection_at_step(-1)
        q_u          = cba.q_hog_u
        lambda_sl    = cba.lambda_sl_at_step(d=d_hog)
        k_lambda     = np.sqrt(lambda_sl / 3.0)
        results2[mode_name].append(dict(A_s_top=A_s_top, xd=xd,
                                        theta_pl=theta_pl,
                                        theta_pl_ec2=theta_pl_ec2,
                                        delta_mid=delta_mid,
                                        q_u=q_u,
                                        lambda_sl=lambda_sl,
                                        k_lambda=k_lambda))
        print(f'  {mode_name:12s}  x/d={xd:.4f}  '
              f'θ_pl={theta_pl:.2f}/{theta_pl_ec2:.2f} mrad (full/±0.6h)  '
              f'δ_mid={delta_mid:.1f} mm  q={q_u:.1f} kN/m  '
              f'λ={lambda_sl:.2f}  k={k_lambda:.3f}')


In [ ]:
fig, (ax_th, ax_lam, ax_d) = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

for mode in ('design', 'mean'):
    r      = results2[mode]
    xd     = np.array([pt['xd']           for pt in r])
    th     = np.array([pt['theta_pl']     for pt in r])
    th_ec  = np.array([pt['theta_pl_ec2'] for pt in r])
    delta  = np.array([pt['delta_mid']    for pt in r])
    lam    = np.array([pt['lambda_sl']    for pt in r])
    c      = COLOR[mode]

    ax_th.plot(xd, th,    'o-',  color=c, lw=2,   ms=7,
               label=fr'$\theta_{{pl}}$ — {mode}, full zone')
    ax_th.plot(xd, th_ec, 's--', color=c, lw=1.5, ms=6,
               label=fr'$\theta_{{pl,EC2}}$ — {mode}, $\pm 0.6h$')
    ax_lam.plot(xd, lam, 'D-',  color=c, lw=2, ms=6,
                label=f'$\lambda$ — {mode}')
    ax_d.plot(xd,  delta, 'o-',  color=c, lw=2,   ms=7,
              label=f'$\delta_{{mid}}$ — {mode}')

ax_lam.axhline(3.0, color='gray', lw=1.2, ls=':', label=r'$\lambda = 3$ (Bild 5.6DE reference)')
ax_lam.text(0.48, 3.05, r'$\lambda = 3$', ha='right', va='bottom', fontsize=9, color='gray')

ax_th.set_ylabel(r'$\theta_{pl}$ [mrad]', fontsize=11)
ax_th.set_title(
    fr'Fixed $A_{{s,bot}}$ = {A_s_bot_fixed:.0f} mm², varying $A_{{s,top}}$ — design vs mean',
    fontsize=10)
ax_th.grid(True, alpha=0.3, linestyle='--')
ax_th.legend(fontsize=9, ncol=2)

ax_lam.set_ylabel(r'$\lambda = a_s / d$  (EC2 §5.6.3)', fontsize=11)
ax_lam.grid(True, alpha=0.3, linestyle='--')
ax_lam.legend(fontsize=9)

ax_d.set_xlabel(r'$x/d$ — hogging section (top bars only)', fontsize=11)
ax_d.set_ylabel(r'$\delta_{mid}$ [mm]  (ULS load)', fontsize=11)
ax_d.grid(True, alpha=0.3, linestyle='--')
ax_d.legend(fontsize=9)

plt.tight_layout()
plt.show()


## Comparison with EC2 Bild 5.6DE

The EC2 (DIN EN 1992-1-1/NA) **Bild 5.6DE** gives empirical allowable plastic
rotations $\theta_{pl,d}$ vs $x_d/d$ for B500B steel at reference shear
slenderness $\lambda = 3$.  Below we overlay the simulation results
($\pm 0.6h$ zone) on the digitised EC2 curve for C12/15–C50/60
(which covers our C30/37) and for C100/115 for reference.

**Cross-section**: $b = 300$ mm, $h = 700$ mm, $d = 650$ mm, C30/37, B500B,
$L = 6$ m.

**Shear slenderness correction**: Our beam at $L = 6$ m has
$\lambda \approx 2.0$–$2.3$ (geometric shear span, see Study 1 above).
The corrected allowable is $\theta_{pl,d}^* = k \cdot \theta_{pl,d}$
with $k = \sqrt{\lambda/3} \approx 0.83$–$0.88$.


In [ ]:
from pathlib import Path

# ── EC2 Bild 5.6DE — digitised from DIN EN 1992-1-1/NA, B500B, λ = 3.0 ──────
# Curves manually read from the chart (grid: Δx/d = 0.05, Δθ = 2 mrad)
xd_ec2    = np.array([0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45])
tpl_c50   = np.array([7.0,  8.5, 13.5, 14.5, 13.0, 11.5, 10.0,  8.5,  7.5,  6.5])  # C12/15–C50/60
tpl_c100  = np.array([7.0,  8.5, 13.0, 12.0, 10.0,  8.5,  7.0,  6.0,  5.0,  4.0])  # C100/115

# Mean k for our Study-1 design-mode results
lam_arr = np.array([pt['lambda_sl'] for pt in results['design']])
k_mean  = float(np.mean(np.sqrt(lam_arr / 3.0)))
lam_mean = float(np.mean(lam_arr))

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# EC2 Bild 5.6DE curves
ax.plot(xd_ec2, tpl_c50,  'k-',  lw=2.5, zorder=5,
        label=r'EC2 Bild 5.6DE — C12/15–C50/60, $\lambda=3$')
ax.plot(xd_ec2, tpl_c100, 'k:',  lw=1.8, zorder=5,
        label=r'EC2 Bild 5.6DE — C100/115, $\lambda=3$')

# k-corrected curve for our λ
tpl_c50_k = tpl_c50 * k_mean
ax.fill_between(xd_ec2, tpl_c50_k, tpl_c50, alpha=0.10, color='gray', zorder=1)
ax.plot(xd_ec2, tpl_c50_k, 'k--', lw=1.8, zorder=5,
        label=fr'EC2 corrected C12/15–C50/60: $k={k_mean:.2f}$  ($\lambda\approx{lam_mean:.1f}$, $L=6$ m)')

# Simulation results (Study 1, symmetric A_s)
for mode in ('design', 'mean'):
    r_   = results[mode]
    xd_s = np.array([pt['xd']           for pt in r_])
    th_s = np.array([pt['theta_pl_ec2'] for pt in r_])
    ax.plot(xd_s, th_s, 'o-', color=COLOR[mode], lw=2, ms=8, zorder=6,
            label=fr'Simulation ({mode}), $\pm 0.6h$ zone')

# EC2 x/d upper limit
ax.axvline(0.45, color='#888888', ls=':', lw=1.0)
ax.text(0.453, 14.8, r'$x/d = 0.45$', fontsize=8, color='#888888', va='top')

ax.set_xlabel(r'$x_d/d$  — neutral-axis depth at hogging section', fontsize=12)
ax.set_ylabel(r'$\theta_{pl}$ [mrad]', fontsize=12)
ax.set_title(
    r'Plastic rotation: simulation ($\pm 0.6h$ zone) vs EC2 Bild 5.6DE'
    '\n'
    r'$b \times h = 300 \times 700$ mm,  C30/37,  B500B,  $L = 6$ m,  symmetric $A_s$',
    fontsize=11)
ax.set_xlim(0.00, 0.50)
ax.set_ylim(0, 16.5)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=9.5, loc='upper right')

# Section parameter text box
info = ('$b=300$ mm,  $h=700$ mm,  $d=650$ mm\\n'
        r'C30/37: $f_{cd}=17.0$ MPa  |  B500B: $f_{yd}=435$ MPa\\n'
        fr'$L=6$ m,  $\lambda\approx{lam_mean:.1f}$,  $k={k_mean:.2f}$')
ax.text(0.02, 0.5, info, transform=ax.transData, fontsize=8.5,
        va='bottom', ha='left',
        bbox=dict(facecolor='white', edgecolor='#aaaaaa', alpha=0.85, pad=4))

plt.tight_layout()

# ── Save to lecture notes fig folder ─────────────────────────────────────────
FIG_OUT = Path('/home/rch/Coding/icc_manuscript/lecture_notes/fig')
fig.savefig(str(FIG_OUT / 'ch10_theta_pl_vs_ec2.png'), dpi=150, bbox_inches='tight')
print(f'Saved: {FIG_OUT}/ch10_theta_pl_vs_ec2.png')
plt.show()


## Curvature profile at ULS — single configuration example

Show κ(x) over the full two-span beam at the ultimate load step for one representative
section (A_s = 1220 mm², x/d ≈ 0.20, design values).  Highlights the
EC2 ±0.6h integration zone and the full hogging zone to connect the abstract
rotation formula to the actual curvature distribution.

In [ ]:
from dataclasses import replace as _replace
from pathlib import Path
FIG_OUT = Path('/home/rch/Coding/icc_manuscript/lecture_notes/fig')

# ── Rebuild one specific configuration ───────────────────────────────────────
A_s_ex  = float(A_s_range[2])   # index 2 → x/d_target = 0.20, A_s ≈ 1220 mm²
triple_ex = ContinuousBeamVarCSModes.for_rc(
    b=b, h=h,
    A_s_bot=A_s_ex, z_s_bot=z_s_bot,
    A_s_top=A_s_ex, z_s_top=z_s_top,
    f_ck=f_ck, f_yk=f_yk,
    L_a=L, L_b=L, n_kappa=100,
)
cba_ex = triple_ex.design

# ── Extract κ(x) profile at ULS ──────────────────────────────────────────────
s         = cba_ex.sweep
idx       = -1
q_ult     = float(s['q'][idx])
M_hog_Nmm = float(s['M_nl_hog'][idx]) * 1e6               # N·mm, signed ≤ 0
La        = float(cba_ex.span_left.L)                      # 6000 mm

cba_q = _replace(cba_ex, q=q_ult)
x_a, M_a, x_b, M_b = cba_q.M_profiles(M_hog_Nmm)
ka = cba_ex.span_left.get_kappa_x_from_M(M_a)
kb = cba_ex.span_right.get_kappa_x_from_M(M_b)

x_all  = np.concatenate([x_a,      x_b + La])
k_all  = np.concatenate([ka,        kb      ])
k_plot = -k_all * 1e3                                      # ‰/mm, hogging positive

# ── Zone boundaries ──────────────────────────────────────────────────────────
x_ec2_lo = La - 0.6 * h
x_ec2_hi = La + 0.6 * h

mask_hog  = k_all < 0
mask_ec2  = (x_all >= x_ec2_lo) & (x_all <= x_ec2_hi) & mask_hog
theta_full = float(np.trapezoid(-k_all[mask_hog], x_all[mask_hog])) * 1e3   # mrad
theta_ec2  = float(np.trapezoid(-k_all[mask_ec2], x_all[mask_ec2])) * 1e3   # mrad

print(f'A_s = {A_s_ex:.0f} mm²   x/d = {cba_ex.xd_hog_u:.3f}   q_u = {q_ult:.1f} kN/m')
print(f'  θ_pl (full zone)  = {theta_full:.2f} mrad')
print(f'  θ_pl (EC2 ±0.6h)  = {theta_ec2:.2f} mrad')

# ── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(x_all, k_plot, color='steelblue', lw=2.0, zorder=4,
        label=r'$\kappa(x)$  (hogging positive)')
ax.axhline(0, color='black', lw=0.8)

ax.fill_between(x_all, 0, k_plot,
                where=mask_hog, alpha=0.18, color='#2ca02c', zorder=2,
                label=fr'Full hogging zone — $\theta_{{pl}}^{{full}} = {theta_full:.2f}$ mrad')

ax.fill_between(x_all, 0, k_plot,
                where=mask_ec2, alpha=0.45, color='#1f77b4', zorder=3,
                label=fr'EC2 $\pm 0.6h$ zone — $\theta_{{pl}}^{{EC2}} = {theta_ec2:.2f}$ mrad')

for xv in (x_ec2_lo, La, x_ec2_hi):
    ax.axvline(xv, color='#1f77b4', ls='--', lw=1.2, alpha=0.8, zorder=1)

y_top = float(np.max(k_plot)) * 0.92
ax.text(x_ec2_lo - 80, y_top, r'$S - 0.6h$', ha='right', va='top', fontsize=8,
        color='#1f77b4')
ax.text(La, y_top, r'$S$', ha='center', va='top', fontsize=8, color='#1f77b4')
ax.text(x_ec2_hi + 80, y_top, r'$S + 0.6h$', ha='left', va='top', fontsize=8,
        color='#1f77b4')

for xs in (0, La, La + L):
    ax.axvline(xs, color='gray', lw=0.5, ls=':', zorder=0)
ax.set_xticks([0, La/2, La, La + L/2, La + L])
ax.set_xticklabels(['A  (x=0)', 'L/2', 'B  (S)', '3L/2', 'C  (x=2L)'])

ax.set_xlabel('Position along beam', fontsize=11)
ax.set_ylabel(r'$\kappa$  [‰/mm]', fontsize=11)
ax.set_title(
    fr'Curvature profile at ULS — $A_s = {A_s_ex:.0f}$ mm²,  '
    fr'$x/d = {cba_ex.xd_hog_u:.3f}$,  $q_u = {q_ult:.1f}$ kN/m  '
    r'(design values, C30/37, B500B)',
    fontsize=11,
)
ax.legend(fontsize=9.5, loc='upper right')
ax.grid(True, alpha=0.3, ls='--')
plt.tight_layout()

fig.savefig(str(FIG_OUT / 'ch10_kappa_profile_example.png'), dpi=150, bbox_inches='tight')
print(f'Saved → {FIG_OUT}/ch10_kappa_profile_example.png')
plt.show()

## Two-panel study figure — θ_pl and δ_mid vs x/d (for slides)

In [ ]:
fig, (ax_th, ax_d) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

for mode, c in [('design', COLOR['design']), ('mean', COLOR['mean'])]:
    r      = results[mode]
    xd     = np.array([pt['xd']           for pt in r])
    th     = np.array([pt['theta_pl']     for pt in r])
    th_ec  = np.array([pt['theta_pl_ec2'] for pt in r])
    delta  = np.array([pt['delta_mid']    for pt in r])

    ax_th.plot(xd, th,    'o-',  color=c, lw=2,   ms=7,
               label=fr'Full zone — {mode}')
    ax_th.plot(xd, th_ec, 's--', color=c, lw=1.5, ms=6,
               label=fr'EC2 $\pm 0.6h$ — {mode}')
    ax_d.plot( xd, delta, 'o-',  color=c, lw=2,   ms=7,
               label=f'{mode}')

ax_th.set_ylabel(r'$\theta_{pl}$  [mrad]', fontsize=11)
ax_th.set_title(
    r'Symmetric $A_s$ — rotation capacity and midspan deflection at ULS'
    '\n'
    fr'$b \times h = {b:.0f} \times {h:.0f}$ mm,  C{f_ck:.0f}/37,  B500B,  $L = {L/1000:.0f}$ m',
    fontsize=10,
)
ax_th.legend(fontsize=9, ncol=2)
ax_th.grid(True, alpha=0.3, ls='--')

ax_d.set_xlabel(r'$x_u/d$  — hogging section', fontsize=11)
ax_d.set_ylabel(r'$\delta_{mid}$  [mm]', fontsize=11)
ax_d.legend(fontsize=9)
ax_d.grid(True, alpha=0.3, ls='--')

# Mark example point (A_s = 1220 mm², x/d ≈ 0.20 design)
for r_, c_ in [(results['design'], COLOR['design'])]:
    pt = r_[2]   # index 2 corresponds to A_s_range[2] = 1220 mm²
    ax_th.scatter([pt['xd']], [pt['theta_pl']],     s=90, color=c_, edgecolors='k',
                  zorder=6, marker='*')
    ax_th.scatter([pt['xd']], [pt['theta_pl_ec2']], s=90, color=c_, edgecolors='k',
                  zorder=6, marker='*')
    ax_d.scatter([pt['xd']],  [pt['delta_mid']],    s=90, color=c_, edgecolors='k',
                 zorder=6, marker='*', label=fr'Example: $A_s={A_s_ex:.0f}$ mm²')
ax_d.legend(fontsize=9)

plt.tight_layout()
fig.savefig(str(FIG_OUT / 'ch10_theta_pl_delta_mid.png'), dpi=150, bbox_inches='tight')
print(f'Saved → {FIG_OUT}/ch10_theta_pl_delta_mid.png')
plt.show()